# មេរៀនទី 13 - ការចងចាំភ្នាក់ងារ


## ការតំឡើង

សៀវភៅកំណត់ត្រានេះបង្ហាញពីរបៀបសាងសង់ភ្នាក់ងារការកក់ដំណើរដំណើរកំសាន្តដោយមាន **អង្គចងចាំមានសុខភាព** ប្រ Using the **Microsoft Agent Framework** (MAF)។

អ្នកនឹងរៀនពីរបៀបដែលអង្គចងចាំភ្នាក់ងារផ្សេងៗគ្នាៈ ការងារ, អំឡុងពេលខ្លី, និងអំឡុងពេលវែង - មានឥទ្ធិពលដល់របៀបដែលភ្នាក់ងាររក្សាទុក និងប្រើប្រាស់ព័ត៌មានក្នុងការពិភាក្សា។

**លក្ខខណ្ឌជាមុន៖**
- គម្រោង Microsoft Foundry ដែលមានគំរូជជែកបានដាក់ចេញ (ឧ. `gpt-4.1-mini`)។
- បានចូលដោយប្រើ Azure CLI — ដំណើរការ `az login` នៅក្នុងបញ្ជាបញ្ជា។
- `AZURE_AI_PROJECT_ENDPOINT` — ចំណុចបញ្ចប់គម្រោង Microsoft Foundry របស់អ្នក។
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ឈ្មោះគំរូដែលអ្នកបានដាក់ចេញ។


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## ប្រភេទនៃអង្គចងចាំភ្នាក់ងារ

ភ្នាក់ងារAIអាចប្រើប្រភេទអង្គចងចាំផ្សេងៗគ្នា ដែលមួយមួយមានគោលបំណងខុសគ្នា៖

### អង្គចងចាំសកម្មភាព
ខ្សែសន្ទនាទាំងមូល — សារដែលប្តូរប្រាស្រ័យក្នុងសម័យតែមួយ។ ភ្នាក់ងារអាចយោងត្រឡប់ទៅឃើញសារមុនៗក្នុងខ្សែសន្ទនាដូចគ្នាដើម្បីរក្សាការស្របគ្នា។ នៅក្នុង MAF នេះត្រូវបានបង្កើតតាមរយៈ **`agent.create_session()`** ដែលទទួលបាន `AgentSession`។

### អង្គចងចាំរយៈពេលខ្លី
ព័ត៌មានដែលនៅរស់រវើកក្នុងរយៈពេលបញ្ហាការងារ ឬសម័យ តែមិនបានផ្ទុកជាការស្ថាពរទេ។ ឧទាហរណ៍ភ្នាក់ងារអាចស្ដុកទុកព័ត៌មានជាច្រើនក្នុងលំដាប់បន្ទាប់បន្សំទាំងបំពេញការពិគ្រោះយោបល់រៀងរាល់ជំហាន ហើយប្រើវាដើម្បីផលិតផែនការចុងក្រោយមួយ។

### អង្គចងចាំរយៈពេលវែង
ចំណូលចិត្ត និងព័ត៌មានដែលនៅរស់រវើក **ក្នុងគ្រប់សម័យផ្សេងៗគ្នា**។ អ្នកប្រើប្រាស់ដែលត្រឡប់មកវិញមិនគួរត្រូវតែបញ្ចេញកំណត់ហេតុអាហារដោះស្រាយរបស់ពួកគេទេ ឬរបៀបធ្វើដំណើរ។ អង្គចងចាំរយៈពេលវែងភាគច្រើនត្រូវបានគាំទ្រដោយហាងក្រៅ​ — ទិន្នន័យក្រដាសទិន្នន័យ ឯកសារ ឬស៊ើវ៉ែវ៉ិចទ័រ — ហើយត្រូវបានបញ្ចូលទៅភ្នាក់ងារតាមរយៈឧបករណ៍។


## អនុស្សារណៈការងារជាមួយព្រឹត្តិការណ៍

របៀបរបស់អនុស្សារណៈដែលសាមញ្ញបំផុតគឺជាសម័យសន្ទនា។ នៅពេលអ្នកផ្ញើសម័យដដែល (បានបង្កើតតាមរយៈ `agent.create_session()`) ទៅការហៅ `agent.run()` ជាប់គ្នា អ្នកជំនួយការមើលឃើញប្រវត្តិសន្ទនាទាំងមូលនៃការសន្ទនានោះ ហើយអាចរំលឹកព័ត៌មានមុនៗបាន។

មកបង្កើតតំណាងដំណើរកំសាន្តមួយ និងបង្ហាញអនុស្សារណៈការងារ។


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

អ្នកតំណាងបាននឹកឃើញថវិកានោះត្រឹមត្រូវព្រោះសារទាំងពីរចែករំលែកកាលវិភាគដដែលគ្នា។ នេះគឺជា **អង្គចងចាំកំពុងដំណើរការ** — វាមាននៅតែមានសម្រាប់ពេលវេលាជីវិតរបស់សម័យ។

### តើមានអ្វីកើតឡើងជាមួយខ្សែថ្មី?

បើយើងបង្កើតសម័យ **ថ្មី**, អ្នកតំណាងមិនមានអង្គចងចាំរក្សានៃការសន្ទនាចាស់ឡើយ៖


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## ប៉ាទ័រសម្រាប់ចងចាំរយៈពេលវែង

ដើម្បីចងចាំចំណង់ចំណូលចិត្តរបស់អ្នកប្រើ **ឆ្ពោះទៅមុខក្នុងសម័យណាមួយ** យើងត្រូវការផ្ទុកមួយដែលបន្តនៅខាងក្រៅខ្សែការពិភាក្សា។ អ្នកភ្នាក់ងារចូលដំណើរការផ្ទុកនេះតាមរយៈ **ឧបករណ៍** — មុខងារដែលវាអាចហៅដើម្បីរក្សាទុក និងយកព័ត៌មានមកវិញ។

ខាងក្រោម យើងអនុវត្តហាងធ្វើតែម្ដងមួយសម្រាប់ចំណូលចិត្តក្នុងអង្គចងចាំ (នៅក្នុងការផលិត អ្នកនឹងប្រើធ្វើជាមូលដ្ឋានទិន្នន័យឬសន្ទស្សន៍ផ្ចិត) ហើយបង្ហាញវាជាឧបករណ៍ដែលអាចប្រើបានសម្រាប់អ្នកភ្នាក់ងារ។

### វិចិត្រសិល្បៈ
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### ស្ថានភាពទី ១ — អ្នកប្រើប្រាស់ដំបូងកក់ការធ្វើដំណើររស់រាយការណ៍អនុស្សាវរីយ៍

សារ៉ាអញ្ជើញមកលើកដំបូង។ អ្នកតំណាងគួរផ្ទុកចិត្តចំណូលចិត្តរបស់នាងតាមឧបករណ៍ និងប្រើវាដើម្បីផ្តល់អនុសាសន៍សណ្ឋាគារ។


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### ស្ថានการณ์ទី 2 — សារ៉ាថំណើរការឡើងវិញ បន្ទាប់ពីប៉ុន្មានសប្តាហ៍

សារ៉ា ចាប់ផ្ដើម **ខ្សែផ្សាយថ្មីមួយ** (ការស្ទិចស្ទួចសម័យថ្មីមួយ)។ ចងចាំកំពុងធ្វើការប្រើប្រាស់ទទេ ប៉ុន្តែឃ្លាំងចំណូលចិត្តរយៈពេលវែង ក៏នៅមានព័ត៌មានរបស់នាង។ ឧត្តមភាគីគួរតែយកវា និងប្រើប្រសូត្តឱ្យផ្ទាល់ខ្លួនដើម្បីផ្គត់ផ្គង់ការណែនាំ។ 


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## សង្ខេប

ក្នុងមេរៀននេះ អ្នកបានរៀនពីប្រភេទអនុស្សាវរីយ៍ពីរបីរបៀបនៃភ្នាក់ងារនិងរបៀបអនុវត្តវានៅជាមួយ Microsoft Agent Framework:

| ប្រភេទអនុស្សាវរីយ៍ | ម៉ាស៊ីនបម្រើបច្ចេកទេស MAF | អាយុកាល |
|---|---|---|
| **កំពុងដំណើរការ** | `agent.create_session()` | សន្ទនាតែមួយ |
| **រយៈពេលខ្លី** | បរិបទដែលបានប្រមូលក្នុងខ្សែបន្ទាត់ | ភារកិច្ច / សម័យតែមួយ |
| **រយៈពេលវែង** | ហាងក្រៅត្រូវបានចូលប្រើតាមរយៈ `@tool` មុខងារ | លើសពីសម័យ |

### ចំនុចសំខាន់
1. **`agent.create_session()`** ផ្តល់នូវអនុស្សាវរីយ៍កំពុងដំណើរការ — ភ្នាក់ងារមើលឃើញប្រវត្តិសន្ទនា ពេញលេញក្នុងសម័យមួយ។
2. **សម័យថ្មីៗបាត់បង់បរិបទ** — គ្មានអនុស្សាវរីយ៍រយៈពេលវែង ភ្នាក់ងារមិនអាចចងចាំសន្ទនាអតីតកាលបានទេ។
3. **មុខងារ `@tool`** ជាជើងភ្នែកដែលភ្ញាក់ — ពួកវាអនុញ្ញាតឱ្យភ្នាក់ងារសន្សំ និងទាញយកព័ត៌មានពីហាងដែលមានអត់កំណត់។
4. **ការផ្ទាល់ខ្លួនរីកចម្រើនជាមួយនឹងពេលវេលា** — ការផ្ទុកចំណូលចិត្តច្រើនទៀត ដើម្បីធ្វើឲ្យការបញ្ជាក់របស់ភ្នាក់ងារល្អប្រសើរឡើង។

### សាកល្បងនៅពិភពពិត
- **សេវាកម្មអតិថិជន**: ចងចាំប្រវត្តិអតិថិជន និងចំណូលចិត្តរបស់ពួកគេ
- **ជំនួយផ្ទាល់ខ្លួន**: រក្សាបរិបទជាទូទៅគ្នាតាមថ្ងៃឬសប្តាហ៍
- **សុខភាព**: តាមដានព័ត៌មាន និងចំណូលចិត្តអ្នកជំងឺ
- **អ៊ី-ខម៉ែ្រស**: ទិញសំភារៈផ្ទាល់ខ្លួនដោយផ្អែកលើប្រវត្តិ

### ជំហានបន្ទាប់
- ជំនួស dict ក្នុងអនុស្សាវរីយ៍ជាមួយមូលដ្ឋានទិនន័យឬហាងវ៉ិចទ័រ (ឧ. Azure AI Search)
- បន្ថែមការបញ្ចប់អនុស្សាវរីយ៍សម្រាប់ព័ត៌មានដែលមានសុពុលភាពពេលវេលា
- បង្កើតប្រព័ន្ធភ្នាក់ងារច្រើនជាមួយអនុស្សាវរីយ៍រួម
- រំលឹកកំណត់ត្រា Cognee សម្រាប់អនុស្សាវរីយ៍គាំទ្រដោយ knowledge graph


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
